# Primary ICU Teams Identification — Full Analysis Notebook

Complete pipeline for **development** (PICU) and **validation** (PICU/NICU/CVICU).

In [ ]:
# --- setup: imports, paths, toggles ---
import sys, os, gc, warnings, logging, pickle
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display, HTML

CODE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))
PYTHON_DIR  = os.path.join(CODE_DIR, 'python')
RESULTS_DIR = os.path.join(CODE_DIR, 'results', 'notebook_outputs')
CACHE_DIR   = os.path.join(CODE_DIR, 'results', 'cache')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,   exist_ok=True)

if PYTHON_DIR not in sys.path:
    sys.path.insert(0, PYTHON_DIR)

from config import DEVELOPMENT_CONFIG, VALIDATION_CONFIG, ROLES
from data_loader  import load_data, preprocess_data
from algorithms   import run_all_algorithms
from evaluation   import (
    compare_with_gold_standard, compute_accuracy,
    compute_return_rate, compute_conditional_accuracy,
)
from bootstrap    import run_dual_bootstrap, get_boot_result

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')

# run-time toggles
N_BOOTSTRAP            = 2500
RUN_BOOTSTRAP          = True   
RUN_PER_UNIT_BOOTSTRAP = True   

# per-cell force-recompute flags; set True to bypass cache and re-run from scratch
FORCE_RECOMPUTE_DEV_LOAD  = True   
FORCE_RECOMPUTE_DEV_ALGO  = True  
FORCE_RECOMPUTE_DEV_BOOT  = True   
FORCE_RECOMPUTE_VAL_LOAD  = True   
FORCE_RECOMPUTE_VAL_UNITS = True   
FORCE_RECOMPUTE_VAL_CROSS = True   


def cache_or_compute(cache_key, compute_fn, force=False):
    """load pickle from cache if available and force=False; else run compute_fn and cache it"""
    cache_path = os.path.join(CACHE_DIR, f'{cache_key}.pkl')
    if not force and os.path.exists(cache_path):
        print(f'  [cache] loading {cache_key} from disk...')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
    print(f'  [cache] computing {cache_key}...')
    result = compute_fn()
    with open(cache_path, 'wb') as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'  [cache] saved {cache_key}')
    return result


print(f'code dir:    {CODE_DIR}')
print(f'results dir: {RESULTS_DIR}')
print(f'cache dir:   {CACHE_DIR}')
print(f'bootstrap: {N_BOOTSTRAP} iters, run={RUN_BOOTSTRAP}')

In [ ]:
# --- helper functions (formatting + bootstrap accessors for main tables) ---

def _pct(v):              return f"{v*100:.1f}%"                if v is not None else 'N/A'
def _pct_ci(v, lo, hi):   return f"{v*100:.1f}% ({lo*100:.1f}\u2013{hi*100:.1f}%)" if None not in (v, lo, hi) else _pct(v)
def _diff_pct(m, lo, hi): return f"{m*100:+.1f}% ({lo*100:+.1f}\u2013{hi*100:+.1f}%)" if None not in (m, lo, hi) else 'N/A'
def _p(v):
    if v is None: return 'N/A'
    return '<0.001' if v < 0.001 else f'{v:.3f}'

def show(df, title='', caption=''):
    """display df with optional title and caption"""
    if title:   display(HTML(f'<h4 style="margin-bottom:3px">{title}</h4>'))
    display(df)
    if caption: display(HTML(f'<p style="font-size:.85em;color:#555;margin-top:3px">{caption}</p>'))
    print()

def _br(boot, level, role):
    """get bootstrap result dict for a level+role; empty dict if missing"""
    return (boot or {}).get(level, {}).get(role) or {}

def _avg_acc(acc_dict):
    vals = [acc_dict.get(r) for r in ROLES if acc_dict.get(r) is not None]
    return float(np.mean(vals)) if vals else None


print('helpers loaded')

---
## Development Phase (PICU, Round 1)

In [ ]:
# --- load development data ---
def _load_dev_data():
    print('  querying BigQuery...')
    activity, ns, redcap = load_data(DEVELOPMENT_CONFIG)
    return {'activity': activity, 'ns': ns, 'redcap': redcap}

print('loading development data...')
_dev_data  = cache_or_compute('dev_load', _load_dev_data, FORCE_RECOMPUTE_DEV_LOAD)
dev_activity = _dev_data['activity']
dev_ns       = _dev_data['ns']
dev_redcap   = _dev_data['redcap']
print(f'  activity: {len(dev_activity):,} rows | note_signers: {len(dev_ns):,} | redcap: {len(dev_redcap):,} patient-days')

In [ ]:
# --- preprocess + run all algorithms ---
def _compute_dev_algo():
    merged, ns, redcap_proc = preprocess_data(dev_activity, dev_ns, dev_redcap, unit=None)
    if merged is None: raise RuntimeError('development preprocessing failed')
    print('  running algorithms (attending: frontline-dependent)...')
    alg      = run_all_algorithms(merged, ns)
    h_comp   = compare_with_gold_standard(alg['heuristic'], redcap_proc)
    lcs_comp = compare_with_gold_standard(alg['lcs'],       redcap_proc)
    if h_comp.empty: raise RuntimeError('comparison failed')
    return {
        'merged': merged, 'ns': ns, 'redcap_proc': redcap_proc,
        'alg': alg, 'h_comp': h_comp, 'lcs_comp': lcs_comp,
        'n': len(h_comp),
        'h_acc':   compute_accuracy(h_comp),
        'lcs_acc': compute_accuracy(lcs_comp),
        'h_rr':    compute_return_rate(h_comp),
        'lcs_rr':  compute_return_rate(lcs_comp),
        'h_ca':    compute_conditional_accuracy(h_comp),
        'lcs_ca':  compute_conditional_accuracy(lcs_comp),
    }

print('development: preprocess + algorithms + evaluation...')
_dev_algo       = cache_or_compute('dev_algo', _compute_dev_algo, FORCE_RECOMPUTE_DEV_ALGO)
dev_merged      = _dev_algo['merged']
dev_ns          = _dev_algo['ns']
dev_redcap_proc = _dev_algo['redcap_proc']
dev_alg         = _dev_algo['alg']
dev_h_comp      = _dev_algo['h_comp']
dev_lcs_comp    = _dev_algo['lcs_comp']
dev_n           = _dev_algo['n']
dev_h_acc       = _dev_algo['h_acc']
dev_lcs_acc     = _dev_algo['lcs_acc']
dev_h_rr        = _dev_algo['h_rr']
dev_lcs_rr      = _dev_algo['lcs_rr']
dev_h_ca        = _dev_algo['h_ca']
dev_lcs_ca      = _dev_algo['lcs_ca']
del dev_activity; gc.collect()

print(f'development: {dev_n} patient-days')
for role in ROLES:
    print(f'  {role}: heur={_pct(dev_h_acc.get(role))} | lcs={_pct(dev_lcs_acc.get(role))}')

In [ ]:
# --- development bootstrap ---
def _compute_dev_boot():
    boot = {}
    if RUN_BOOTSTRAP:
        print(f'  dual bootstrap ({N_BOOTSTRAP} iters)...')
        boot = run_dual_bootstrap(dev_h_comp, dev_lcs_comp, n_iter=N_BOOTSTRAP)
    return {'boot': boot}

print('development bootstrap...')
_dev_boot = cache_or_compute('dev_boot', _compute_dev_boot, FORCE_RECOMPUTE_DEV_BOOT)
dev_boot  = _dev_boot['boot']
print('done.')

---
## Validation Phase (PICU / NICU / CVICU, Round 2)

In [ ]:
# --- load validation data ---
def _load_val_data():
    print('  querying BigQuery...')
    activity, ns, redcap = load_data(VALIDATION_CONFIG)
    return {'activity': activity, 'ns': ns, 'redcap': redcap}

print('loading validation data...')
_val_data    = cache_or_compute('val_load', _load_val_data, FORCE_RECOMPUTE_VAL_LOAD)
val_activity = _val_data['activity']
val_ns       = _val_data['ns']
val_redcap   = _val_data['redcap']
print(f'  activity: {len(val_activity):,} | note_signers: {len(val_ns):,} | redcap: {len(val_redcap):,}')
print(f'  units: {val_redcap["unit"].value_counts().to_dict()}')

In [ ]:
# --- per-unit analysis loop ---
def _compute_val_units():
    results = {}
    for unit in VALIDATION_CONFIG['units']:
        print(f'\n--- {unit} ---')
        u_merged, u_ns, u_redcap = preprocess_data(val_activity, val_ns, val_redcap, unit=unit)
        if u_merged is None:
            results[unit] = {'status': 'error', 'error': 'preprocessing failed'}; continue

        u_alg      = run_all_algorithms(u_merged, u_ns)
        u_h_comp   = compare_with_gold_standard(u_alg['heuristic'], u_redcap)
        u_lcs_comp = compare_with_gold_standard(u_alg['lcs'],       u_redcap)
        if u_h_comp.empty:
            results[unit] = {'status': 'error', 'error': 'comparison failed'}; continue

        u_n        = len(u_h_comp)
        u_h_acc    = compute_accuracy(u_h_comp)
        u_lcs_acc  = compute_accuracy(u_lcs_comp)
        u_h_rr,  u_lcs_rr  = compute_return_rate(u_h_comp),           compute_return_rate(u_lcs_comp)
        u_h_ca,  u_lcs_ca  = compute_conditional_accuracy(u_h_comp),  compute_conditional_accuracy(u_lcs_comp)

        u_boot = {}
        if RUN_BOOTSTRAP and RUN_PER_UNIT_BOOTSTRAP:
            print(f'  bootstrap {unit}...')
            u_boot = run_dual_bootstrap(u_h_comp, u_lcs_comp, n_iter=N_BOOTSTRAP)

        print(f'  n={u_n} | heur nurse={_pct(u_h_acc.get("nurse"))} fl={_pct(u_h_acc.get("frontline"))} att={_pct(u_h_acc.get("attending"))}')
        print(f'         | lcs  nurse={_pct(u_lcs_acc.get("nurse"))} fl={_pct(u_lcs_acc.get("frontline"))} att={_pct(u_lcs_acc.get("attending"))}')

        results[unit] = {
            'status': 'success', 'n': u_n,
            'merged_data': u_merged, 'redcap': u_redcap,
            'heuristic': {'acc': u_h_acc, 'comp': u_h_comp, 'rr': u_h_rr, 'ca': u_h_ca},
            'lcs':       {'acc': u_lcs_acc, 'comp': u_lcs_comp, 'rr': u_lcs_rr, 'ca': u_lcs_ca},
            'boot': u_boot,
        }
        gc.collect()
    return {'unit_results': results}

print('per-unit validation analysis...')
_val_units       = cache_or_compute('val_units', _compute_val_units, FORCE_RECOMPUTE_VAL_UNITS)
unit_results     = _val_units['unit_results']
successful_units = [u for u, r in unit_results.items() if r.get('status') == 'success']
print(f'\nsuccessful units: {successful_units}')

In [ ]:
# --- cross-unit combined analysis ---
def _compute_val_cross():
    all_h_comps, all_lcs_comps = [], []
    all_merged, all_redcap = [], []
    for unit in successful_units:
        r  = unit_results[unit]
        hc = r['heuristic']['comp'].copy(); hc['unit'] = unit
        lc = r['lcs']['comp'].copy();       lc['unit'] = unit
        md = r['merged_data'].copy();       md['unit'] = unit
        rc = r['redcap'].copy()
        all_h_comps.append(hc); all_lcs_comps.append(lc)
        all_merged.append(md);  all_redcap.append(rc)

    c_h_comp   = pd.concat(all_h_comps,   ignore_index=True)
    c_lcs_comp = pd.concat(all_lcs_comps, ignore_index=True)
    c_merged   = pd.concat(all_merged,    ignore_index=True)
    c_redcap   = pd.concat(all_redcap,    ignore_index=True)

    c_h_acc         = compute_accuracy(c_h_comp)
    c_lcs_acc       = compute_accuracy(c_lcs_comp)
    c_h_rr          = compute_return_rate(c_h_comp)
    c_lcs_rr        = compute_return_rate(c_lcs_comp)
    c_h_ca          = compute_conditional_accuracy(c_h_comp)
    c_lcs_ca        = compute_conditional_accuracy(c_lcs_comp)

    c_boot = {}
    if RUN_BOOTSTRAP:
        print(f'  cross-unit dual bootstrap ({N_BOOTSTRAP} iters)...')
        c_boot = run_dual_bootstrap(c_h_comp, c_lcs_comp, n_iter=N_BOOTSTRAP)

    return {
        'h_comp': c_h_comp, 'lcs_comp': c_lcs_comp, 'merged': c_merged, 'redcap': c_redcap,
        'n': len(c_h_comp),
        'h_acc': c_h_acc, 'lcs_acc': c_lcs_acc,
        'h_rr':  c_h_rr,  'lcs_rr':  c_lcs_rr,
        'h_ca':  c_h_ca,  'lcs_ca':  c_lcs_ca,
        'boot': c_boot,
    }

print('cross-unit combined analysis...')
_val_cross           = cache_or_compute('val_cross', _compute_val_cross, FORCE_RECOMPUTE_VAL_CROSS)
cross_h_comp         = _val_cross['h_comp']
cross_lcs_comp       = _val_cross['lcs_comp']
cross_merged         = _val_cross['merged']
cross_redcap         = _val_cross['redcap']
cross_n              = _val_cross['n']
cross_h_acc          = _val_cross['h_acc']
cross_lcs_acc        = _val_cross['lcs_acc']
cross_h_rr           = _val_cross['h_rr']
cross_lcs_rr         = _val_cross['lcs_rr']
cross_h_ca           = _val_cross['h_ca']
cross_lcs_ca         = _val_cross['lcs_ca']
cross_boot           = _val_cross['boot']

print(f'cross-unit: {cross_n} patient-days across {len(successful_units)} units')
for role in ROLES:
    print(f'  {role}: heur={_pct(cross_h_acc.get(role))} | lcs={_pct(cross_lcs_acc.get(role))}')

---
## Main Paper Tables

In [ ]:
# --- TABLE 1: Development Performance (PICU) ---
# rows: Nurse, Frontline, Attending, Mean accuracy (across roles)
# cols: n, Heuristic Accuracy (95% CI), LCS Accuracy (95% CI), Delta (95% CI), p-value
# "Mean accuracy" = unweighted mean of the 3 per-role rates; macro=micro here since each
# patient-day contributes exactly once to every role's denominator
def build_by_role_table(n, h_acc, lcs_acc, boot):
    rows = []
    for role in ROLES:
        bpd = _br(boot, 'patient_day', role)
        rows.append({
            'Role':                          role.capitalize(),
            'n':                             n,
            'Heuristic Accuracy (95% CI)':   _pct_ci(h_acc.get(role),   bpd.get('h_ci_lower'),   bpd.get('h_ci_upper')),
            'LCS Accuracy (95% CI)':         _pct_ci(lcs_acc.get(role), bpd.get('lcs_ci_lower'), bpd.get('lcs_ci_upper')),
            'Δ Accuracy (95% CI)':           _diff_pct(bpd.get('mean_diff'),  bpd.get('ci_lower'),  bpd.get('ci_upper')),
            'p-value':                       _p(bpd.get('p_value')),
        })
    # mean accuracy across roles (nurse + frontline + attending)
    h_avg, lcs_avg = _avg_acc(h_acc), _avg_acc(lcs_acc)
    diff_avg = (lcs_avg - h_avg) if None not in (h_avg, lcs_avg) else None
    rows.append({'Role': 'Mean accuracy (across roles)', 'n': n,
                 'Heuristic Accuracy (95% CI)': _pct(h_avg),
                 'LCS Accuracy (95% CI)': _pct(lcs_avg),
                 'Δ Accuracy (95% CI)': f"{diff_avg*100:+.1f}%" if diff_avg is not None else 'N/A',
                 'p-value': '\u2014'})
    return pd.DataFrame(rows)

_ci_caption = ('Accuracy = proportion of patient-days correct. 95% CIs from 2500 bootstrap samples '
               '(patient-day resampling). Mean accuracy = unweighted mean of per-role accuracies.')

table1_dev = build_by_role_table(dev_n, dev_h_acc, dev_lcs_acc, dev_boot)
show(table1_dev, title='Table 1 \u2014 Development Performance (PICU)', caption=_ci_caption)

In [ ]:
# --- TABLE 2: Validation Performance by Unit ---
# rows grouped by unit: Nurse, Frontline, Attending, Mean accuracy (across roles)
# "All Units (pooled)" = patient-days from all units concatenated; larger units contribute more
# (micro-average across units). Per-unit rows are also micro-averaged across roles (macro=micro since each patient-day contributes once per role).
def build_by_unit_table(unit_results_dict):
    rows = []
    for unit in (successful_units + ['All Units (pooled)']):
        if unit == 'All Units (pooled)':
            n, h_acc, lcs_acc, boot = cross_n, cross_h_acc, cross_lcs_acc, cross_boot
        else:
            r = unit_results_dict[unit]
            if r.get('status') != 'success': continue
            n, h_acc, lcs_acc, boot = r['n'], r['heuristic']['acc'], r['lcs']['acc'], r.get('boot', {})
        h_avg, lcs_avg = _avg_acc(h_acc), _avg_acc(lcs_acc)
        diff_avg = (lcs_avg - h_avg) if None not in (h_avg, lcs_avg) else None
        # per-role rows
        for role in ROLES:
            bpd = _br(boot, 'patient_day', role)
            rows.append({
                'Unit': unit, 'Role': role.capitalize(), 'n': n,
                'Heuristic Accuracy (95% CI)':   _pct_ci(h_acc.get(role),   bpd.get('h_ci_lower'),   bpd.get('h_ci_upper')),
                'LCS Accuracy (95% CI)':         _pct_ci(lcs_acc.get(role), bpd.get('lcs_ci_lower'), bpd.get('lcs_ci_upper')),
                'Δ (95% CI)':                    _diff_pct(bpd.get('mean_diff'),  bpd.get('ci_lower'),  bpd.get('ci_upper')),
                'p-value':                       _p(bpd.get('p_value')),
            })
        # mean accuracy across roles (nurse + frontline + attending)
        rows.append({'Unit': unit, 'Role': 'Mean accuracy (across roles)', 'n': n,
                     'Heuristic Accuracy (95% CI)': _pct(h_avg),
                     'LCS Accuracy (95% CI)': _pct(lcs_avg),
                     'Δ (95% CI)': f"{diff_avg*100:+.1f}%" if diff_avg is not None else 'N/A',
                     'p-value': '\u2014'})
    return pd.DataFrame(rows)

table2 = build_by_unit_table(unit_results)
show(table2,
     title='Table 2 \u2014 Validation Performance by Unit',
     caption='Accuracy = proportion of patient-days correct. 95% CIs from 2500 bootstrap samples '
             '(patient-day resampling). '
             'Mean accuracy = unweighted mean of per-role accuracies (macro = micro since each '
             'patient-day contributes once per role). All Units (pooled) = patient-days concatenated '
             'across PICU, NICU, and CVICU; larger units contribute proportionally more.')

---
## Descriptive Statistics

In [ ]:
# --- descriptive statistics ---

def describe_dataset(comp_df, merged_df, phase, unit_label):
    """total patient-days, unique encounters, unique HCWs, observation dates"""
    n_pd     = len(comp_df)
    n_enc    = comp_df['CSN'].nunique()
    n_dates  = comp_df['access_date'].nunique()
    n_hcws   = merged_df['USER_ID'].nunique()
    n_hcw_type = merged_df.groupby('prov_type')['USER_ID'].nunique().to_dict()
    return {
        'Phase': phase, 'Unit': unit_label,
        'N patient-days':       n_pd,
        'N encounters (CSNs)':  n_enc,
        'Avg patient-days/enc': round(n_pd/n_enc,2) if n_enc > 0 else None,
        'N unique observation dates': n_dates,
        'N unique HCWs in pool':      n_hcws,
        'HCW types': str({k: v for k,v in sorted(n_hcw_type.items(), key=lambda x: -x[1])[:5]}),
    }

desc_rows = [describe_dataset(dev_h_comp, dev_merged, 'Development', 'PICU')]
for u in successful_units:
    r = unit_results[u]
    desc_rows.append(describe_dataset(r['heuristic']['comp'], r['merged_data'], 'Validation', u))
desc_rows.append(describe_dataset(cross_h_comp, cross_merged, 'Validation', 'All Units'))

desc_table = pd.DataFrame(desc_rows)
show(desc_table, title='Descriptive Statistics')

# bootstrap CI summary
print('\n--- Bootstrap CI (patient-day resampling) ---')
for role in ROLES:
    bpd = _br(cross_boot, 'patient_day', role)
    pd_ci = _diff_pct(bpd.get('mean_diff'),  bpd.get('ci_lower'),  bpd.get('ci_upper'))
    print(f'  {role}: {pd_ci} (p={_p(bpd.get("p_value"))})')

---
## Save All Tables to CSV

In [ ]:
# --- save all tables ---
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

save_map = {
    # main tables: table1=dev (PICU), table2=val (per-unit + pooled)
    f'table1_dev_picu_{ts}.csv':                   table1_dev,
    f'table2_val_by_unit_{ts}.csv':                table2,
    # descriptive
    f'descriptive_stats_{ts}.csv':                desc_table
}

for fname, df in save_map.items():
    if df is None or (hasattr(df, 'empty') and df.empty):
        print(f'skip (empty): {fname}'); continue
    path = os.path.join(RESULTS_DIR, fname)
    df.to_csv(path, index=False)
    print(f'saved: {fname}  ({len(df)} rows)')

print(f'\nall outputs: {RESULTS_DIR}')